# Estimasi, Pola, dan Proyeksi Emisi Karbon Hutan Indonesia di Tingkat Kabupaten/Kota

**Data mining dengan Gradient Boosting (target intensitas emisi), SHAP, klasterisasi, deteksi anomali, proyeksi 2026-2030, dan dashboard Streamlit**

| | |
|---|---|
| **Tim / Peserta** | `[isi nama tim]` |
| **Lomba** | `[isi nama lomba]` |
| **Sumber data** | Global Forest Watch (GFW) - `IDN.xlsx`, rilis v20260427, 2001-2025, tingkat kabupaten/kota |
| **Cakupan** | 497 kabupaten/kota x 25 tahun = 12.425 observasi |
| **Bahasa / Tools** | Python 3, pandas, scikit-learn, SHAP, SciPy, Streamlit |

## Ringkasan Eksekutif

| Pertanyaan | Temuan utama |
|---|---|
| **Seberapa besar emisinya?** | Emisi kotor rata-rata **943 Mt CO2e/tahun**, serapan **594 Mt**, sehingga **net flux nasional +349 Mt CO2e/tahun** (net *source*). Namun **308 dari 497 kabupaten** adalah *net sink*. |
| **Siapa penyumbang terbesar?** | Emisi sangat terkonsentrasi: **86 kabupaten (17%)** menghasilkan **80%** emisi kumulatif (prinsip Pareto). |
| **Mana yang lebih berbahaya, hutan primer atau bukan?** | Kehilangan hutan primer melepas sekitar **1.170 Mg CO2e/ha** vs **629 Mg CO2e/ha** untuk non-primer (sekitar **1,9x**), indikatif (NNLS). |
| **Seberapa akurat modelnya?** | Pada validasi tanpa kebocoran, **MAE turun 55%** (tahun mendatang) dan **23%** (kabupaten tak terlihat) dibanding baseline Ridge. R² tahun uji 2022-2025 = **0,98** (model WARM). |
| **Apakah ada tipologi wilayah?** | Ya, **5 klaster** kabupaten berdasarkan komposisi pemicu, kerapatan karbon, tren, dan status net flux (kekuatan struktur klaster: sedang, silhouette 0,257). |
| **Kapan emisi mendadak melonjak?** | **20 kabupaten-tahun** anomali, paling banyak pada **2015 (6)** dan **2016 (5)**. |
| **Bagaimana proyeksinya?** | Proyeksi 2026-2030 per kabupaten dengan **interval 80%**, diuji lewat *rolling-origin backtest* terhadap baseline naif. |

> Seluruh angka di atas berasal dari eksekusi notebook ini pada data GFW. Atribusi per pemicu bersifat **korelatif, bukan kausal**; keterbatasan lengkap ada di bagian akhir.

## Daftar Isi

**Bagian I - Penjelasan Project**
**Bagian II - Penjelasan Metodologi**
**Bagian III - Eksekusi (setiap bagian: apa ini, penjelasan, kode, hasil, interpretasi)**

| No | Bagian | Fase CRISP-DM |
|---|---|---|
| 0 | Persiapan lingkungan & konfigurasi | - |
| 1 | Pemuatan data, audit, dan *feature engineering* | Data Understanding & Preparation |
| 2 | Eksplorasi data (EDA) | Data Understanding |
| 3 | Pemodelan & validasi tanpa kebocoran | Modeling & Evaluation |
| 4 | Lapisan data mining: Pareto, NNLS, net flux, hotspot, SHAP, klaster, anomali, proyeksi | Modeling & Evaluation |
| 5 | Deployment: dashboard Streamlit | Deployment |
| 6 | Kesimpulan, rekomendasi, keterbatasan, dan pengembangan lanjutan | Evaluation |
| 7 | Lampiran: artefak keluaran & referensi | - |

# BAGIAN I - PENJELASAN PROJECT

## 1. Latar Belakang
Indonesia berkomitmen mencapai target **FOLU Net Sink 2030**, yaitu sektor kehutanan dan penggunaan lahan yang menyerap lebih banyak karbon daripada yang dilepaskannya. Tantangannya, **laju kehilangan tutupan pohon dan emisi karbon sangat bervariasi antarwilayah** dan dipicu faktor yang berbeda-beda: pertanian, kebakaran, pembalakan, tambang, hingga infrastruktur. Kebijakan yang seragam tidak efisien; pemerintah daerah dan pemangku kepentingan perlu tahu **wilayah mana yang paling menentukan, apa pemicunya, dan ke mana arah emisi berikutnya**.

## 2. Rumusan Masalah
1. Berapa **emisi kotor tahunan** karbon hutan di setiap kabupaten/kota, dan bisakah diestimasi dari luas kehilangan tutupan per pemicu serta kondisi hutannya?
2. **Pola apa** yang tersembunyi: seberapa terkonsentrasi emisi, adakah tipologi wilayah, kapan terjadi lonjakan, dan di mana ada "karbon tersembunyi" (mis. gambut)?
3. Bagaimana **proyeksi 5 tahun ke depan** dan seberapa pasti proyeksi tersebut?

## 3. Tujuan Data Mining
| # | Tujuan | Jenis tugas |
|---|---|---|
| 1 | Mengestimasi emisi kotor tahunan per kabupaten | Regresi (prediktif) |
| 2 | Menemukan konsentrasi, tipologi, anomali, dan hotspot | Deskriptif: Pareto, NNLS, klasterisasi, deteksi anomali |
| 3 | Memproyeksikan emisi 2026-2030 dengan interval ketidakpastian | *Forecasting* langsung (h = 1..5) |
| 4 | Menyediakan alat bantu keputusan untuk simulasi skenario | *Decision support system* (Streamlit) |

## 4. Kriteria Keberhasilan
* Akurasi pada validasi yang **tidak bocor**: *future years* (tahun yang belum dilihat model) dan *kabupaten tak terlihat*.
* **MAE lebih rendah** daripada baseline sederhana (Ridge untuk regresi; *naive* untuk proyeksi).
* Ada **interval ketidakpastian**, bukan hanya angka tunggal.
* Pernyataan **keterbatasan yang jujur**: atribusi per pemicu bersifat korelatif, bukan kausal.

## 5. Data yang Digunakan
**Global Forest Watch** - `IDN.xlsx`, rilis v20260427, periode 2001-2025, tingkat *Subnational 2* (kabupaten/kota), ambang tutupan pohon 30%.

| Sheet | Isi |
|---|---|
| `Subnational 2 tree cover loss` | Luas kehilangan tutupan (ha) per tahun + luas wilayah & tutupan awal |
| `Subnational 2 primary loss` | Kehilangan hutan primer (mulai 2002) |
| `Subnational 2 drivers` | Kehilangan tutupan (ha) per **7 pemicu** per tahun |
| `Subnational 2 primary drivers` | Kehilangan hutan primer per pemicu utama |
| `Subnational 2 carbon data` | Stok & densitas biomassa 2000, **emisi kotor tahunan**, serapan, net flux rata-rata |

**7 pemicu:** *Hard commodities* (tambang), *Logging*, *Other natural disturbances*, *Permanent agriculture*, *Settlements & Infrastructure*, *Shifting cultivation*, *Wildfire*.

## 6. Nilai Tambah Project
| Aspek | Pendekatan di notebook ini |
|---|---|
| Validasi | Split **temporal** dan **GroupKFold per kabupaten**, bukan *random split* yang membocorkan tahun-tahun kabupaten yang sama |
| Target model | Memprediksi **log-intensitas emisi** (Mg CO2e per ha hilang), lalu dikalikan luas hilang, sehingga tidak jenuh pada skenario *what-if* berskala besar |
| Karbon tersembunyi | Fitur `hist_intensity` (riwayat intensitas kabupaten itu sendiri, **hanya dari tahun sebelumnya**) sebagai proksi gambut/biomassa bawah tanah |
| Ketidakpastian | Model kuantil (P10/P90) dan interval empiris dari *backtest* |
| Pembanding | Setiap model dibandingkan dengan baseline; proyeksi dibandingkan dengan *naive* |
| Kejujuran | Audit data, kontrol kebocoran, dan pernyataan keterbatasan eksplisit |
| Penggunaan | Dashboard Streamlit untuk simulasi skenario |

# BAGIAN II - PENJELASAN METODOLOGI YANG DIGUNAKAN

## 1. Kerangka Kerja: CRISP-DM
Project mengikuti **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*).

```
Business        Data            Data               Modeling &          Evaluation          Deployment
Understanding -> Understanding -> Preparation -----> Validasi jujur ----> (keterbatasan) ---> Dashboard
(Bagian I)      (Bagian 1-2)    (Bagian 1)         (Bagian 3-4)        (Bagian 6)          (Bagian 5)
                                  |                    |
                                  +-- audit & kontrol  +-- split temporal + GroupKFold
                                      kebocoran            + baseline + interval
```

## 2. Formulasi Model: Emisi = Luas Hilang x Intensitas
GFW menghitung emisi dari **luas hilang x biomassa**. Maka model tidak memprediksi emisi mentah, melainkan **intensitas**:

$$\text{intensitas} = \log\frac{E + 1}{\text{loss\_ha} + 1}, \qquad \hat{E} = (\text{loss\_ha}+1)\cdot e^{\hat{\text{intensitas}}} - 1$$

Dengan begitu prediksi **skala dengan luas hektar**, tidak jenuh saat pengguna mensimulasikan kehilangan tutupan yang besar.

## 3. Rancangan Validasi (Anti-Kebocoran)
| Skenario | Cara | Menjawab pertanyaan |
|---|---|---|
| **Future years** | Latih tahun <= 2021, uji 2022-2025 | Apakah model bisa memprediksi tahun yang belum terjadi? |
| **Kabupaten tak terlihat** | `GroupKFold(5)` per kabupaten | Apakah model bisa diterapkan ke kabupaten baru? |
| **Rolling-origin backtest** | Origin 2012/2015/2018/2020, latih hanya dengan informasi <= origin | Apakah proyeksi h=1..5 lebih baik dari *naive*? |

**Dua set fitur:** **COLD** (kabupaten baru, tanpa riwayat) dan **WARM** (menambah `hist_intensity`, dipakai di aplikasi).

**Kontrol kebocoran (*leakage*):**
1. *Net flux* dan rata-rata emisi tahunan **dihitung dari target**, sehingga **tidak dipakai sebagai fitur**.
2. `cum_loss_prior_ha`, `remaining_*`, dan `hist_intensity` hanya memakai informasi **sebelum tahun t**.
3. Tidak ada *random split* pada data panel.
4. *Backtest* proyeksi hanya memakai data yang tersedia pada tahun origin.

## 4. Metrik Evaluasi
| Metrik | Fungsi | Catatan |
|---|---|---|
| **MAE** | Metrik utama (Mg CO2e per kabupaten-tahun) | Tahan terhadap kabupaten raksasa |
| **R²** | Metrik pendamping | Didominasi segelintir kabupaten besar, **dibaca bersama MAE** |
| **Silhouette** | Kualitas klaster | Menentukan jumlah klaster K |

## 5. Teknik Data Mining yang Dipakai
| Teknik | Pertanyaan | Keluaran |
|---|---|---|
| Regresi: Ridge, Random Forest, HistGradientBoosting | Berapa emisi tahunan? | `model_emission.joblib`, `metrics.json` |
| Permutation importance | Fitur apa yang menjelaskan intensitas? | `feature_importance.csv/.png` |
| Analisis Pareto | Seberapa terkonsentrasi emisi? | `pareto.csv` |
| NNLS (*non-negative least squares*) | Berapa Mg CO2e per ha untuk tiap pemicu? | `driver_intensity.csv` |
| Net flux | Mana net *source* dan net *sink*? | `net_table.csv` |
| Rasio intensitas vs ekspektasi biomassa | Di mana "karbon tersembunyi"? | `intensity_hotspots.csv` |
| SHAP | Pemicu mana berpengaruh pada model interpretasi? | `shap_driver.csv/.png` |
| K-Means + PCA + Silhouette | Apa tipologi kabupaten? | `cluster_table.csv`, `cluster_summary.csv` |
| Robust z-score (median/MAD) | Kapan emisi melonjak tak wajar? | `anomalies.csv` |
| Direct multi-horizon forecasting + backtest | Berapa emisi 2026-2030? | `forecast_backtest.csv`, `forecast_2026_2030.csv` |

# BAGIAN III - EKSEKUSI

Setiap bagian di bawah memakai format yang sama:

> **Ini bagian apa** -> **Penjelasan** -> **Kode** -> **Apa yang didapat** -> **Interpretasi**

Jalankan sel **berurutan dari atas** (*Run All*). Letakkan `IDN.xlsx` di folder kerja (di Colab: unggah lewat panel *Files*).

## 0. Persiapan Lingkungan & Konfigurasi

**Ini bagian apa:** pemasangan pustaka dan pengaturan awal notebook.

**Penjelasan:** `shap` dan `openpyxl` dipasang sekali (khusus Colab). Seluruh *import* dikumpulkan di satu sel agar bagian berikutnya fokus pada logika analisis. Peringatan numerik `LinAlgWarning` dari Ridge dibungkam karena fitur memang berskala sangat berbeda (hektar vs rasio) dan tidak memengaruhi hasil.

In [ ]:
!pip -q install shap openpyxl   # hanya perlu di Colab; IDN.xlsx harus diunggah ke folder kerja

In [ ]:
import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn
from scipy.linalg import LinAlgWarning
from scipy.optimize import nnls
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=LinAlgWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | shap {shap.__version__}")

#### Apa yang didapat
Baris versi pustaka yang dipakai.

#### Penjelasan
Versi dicetak demi **reproduksibilitas**. Angka dapat bergeser sedikit bila versi pustaka berbeda. Jika `model_emission.joblib` gagal dimuat di aplikasi karena beda versi scikit-learn, jalankan ulang bagian 3.

## 1. Pemuatan Data, Audit, dan Feature Engineering

**Ini bagian apa:** *Data Understanding* + *Data Preparation*. Dari lima sheet GFW dibangun satu **tabel panel** (kabupaten x tahun).

**Penjelasan:**
* Panel dibangun dari sheet karbon (**497 kabupaten x 25 tahun = 12.425 baris**), lalu *LEFT join* ke tabel pemicu dan hutan primer. Pendekatan lama dengan *inner join* membuang 3 kabupaten.
* Kabupaten-tahun tanpa catatan pemicu diisi **0** (tidak ada catatan = tidak ada kehilangan terekam). Audit menunjukkan baris tersebut hanya memuat **3,7e-7** dari total emisi, sehingga pengisian ini aman.
* **Fitur baru:**
  * *Kondisi hutan dinamis* - `cum_loss_prior_ha`, `remaining_extent_ha`, `remaining_frac`, `remaining_carbon_Mg_C`
  * *Struktur pemicu* - `agri_loss_ratio`, `wildfire_ratio`
  * *Hutan primer* - `primary_share` (bagian kehilangan yang berkarbon tinggi dan sulit pulih)
  * *Struktur wilayah* - `forest_cover_frac`, `loss_extent_ratio`
  * *Karbon tersembunyi* - `hist_intensity`: emisi per ha hilang milik kabupaten itu sendiri pada tahun-tahun **sebelumnya** (bebas kebocoran)
* **Kontrol kebocoran:** *net flux* dan rata-rata emisi tahunan dihitung dari target sehingga **tidak dijadikan fitur** (`gross_avg_Mg_CO2e_yr` dibuang dari panel).

In [ ]:
XLSX = "IDN.xlsx"
TH = 30                      # canopy-density threshold (30% = GFW standard for forest)
YEARS = list(range(2001, 2026))
KEY = ["subnational1", "subnational2"]
DRIVERS = ["Hard commodities", "Logging", "Other natural disturbances", "Permanent agriculture",
           "Settlements & Infrastructure", "Shifting cultivation", "Wildfire"]

x = pd.ExcelFile(XLSX)
rd = lambda s: pd.read_excel(x, sheet_name=s)
tc = rd("Subnational 2 tree cover loss"); tc = tc[tc["threshold"] == TH]
pl = rd("Subnational 2 primary loss");    pl = pl[pl["threshold"] == TH]
dr = rd("Subnational 2 drivers");         dr = dr[dr["threshold"] == TH]
pdr = rd("Subnational 2 primary drivers"); pdr = pdr[pdr["threshold"] == TH]
cb = rd("Subnational 2 carbon data");     cb = cb[cb["umd_tree_cover_density_2000__threshold"] == TH]

audit = {"kabupaten_carbon": int(len(cb)), "kabupaten_drivers": int(dr.groupby(KEY).ngroups)}

# ---- long tables ------------------------------------------------------------------------------
def melt_years(df, prefix, name, years):
    cols = [f"{prefix}{y}" for y in years if f"{prefix}{y}" in df.columns]
    m = df.melt(id_vars=KEY, value_vars=cols, var_name="c", value_name=name)
    m["year"] = m["c"].str.extract(r"(\d{4})").astype(int)
    return m.drop(columns="c")

em_cols = [f"gfw_forest_carbon_gross_emissions_{y}__Mg_CO2e" for y in YEARS]
em = cb.melt(id_vars=KEY, value_vars=em_cols, var_name="c", value_name="gross_emissions_Mg_CO2e")
em["year"] = em["c"].str.extract(r"(\d{4})").astype(int); em = em.drop(columns="c")

tcl = melt_years(tc, "tc_loss_ha_", "tc_loss_ha", YEARS)
pll = melt_years(pl, "tc_loss_ha_", "primary_loss_ha", YEARS[1:])          # primary loss starts 2002
drp = dr.pivot_table(index=KEY + ["year"], columns="driver", values="tc_loss_ha", fill_value=0).reset_index()
for d in DRIVERS:
    if d not in drp: drp[d] = 0.0
pdp = pdr.pivot_table(index=KEY + ["year"], columns="driver", values="tc_loss_ha", fill_value=0).reset_index()
pdp = pdp.rename(columns={"Permanent agriculture": "primary_agri_ha", "Wildfire": "primary_fire_ha", "Logging": "primary_logging_ha"})
pdp = pdp[KEY + ["year", "primary_agri_ha", "primary_fire_ha", "primary_logging_ha"]]

static = tc[KEY + ["area_ha", "extent_2000_ha", "extent_2010_ha"]].merge(
    cb[KEY + ["umd_tree_cover_extent_2000__ha", "gfw_aboveground_carbon_stocks_2000__Mg_C",
              "avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1",
              "gfw_forest_carbon_gross_removals__Mg_CO2_yr-1", "gfw_forest_carbon_net_flux__Mg_CO2e_yr-1",
              "gfw_forest_carbon_gross_emissions__Mg_CO2e_yr-1"]], on=KEY, how="outer")

# ---- panel: start from the CARBON sheet (497 kabupaten x 25 years) and LEFT-join the rest ------
df = em.merge(static, on=KEY, how="left").merge(tcl, on=KEY + ["year"], how="left")
df = df.merge(drp, on=KEY + ["year"], how="left").merge(pll, on=KEY + ["year"], how="left")
df = df.merge(pdp, on=KEY + ["year"], how="left")
audit["rows_panel"] = int(len(df))
audit["rows_without_driver_record"] = int(df[DRIVERS].isna().all(axis=1).sum())
df[DRIVERS] = df[DRIVERS].fillna(0.0)                                  # no record = no recorded loss
df[["primary_agri_ha", "primary_fire_ha", "primary_logging_ha"]] = df[["primary_agri_ha", "primary_fire_ha", "primary_logging_ha"]].fillna(0.0)
df["tc_loss_ha"] = df["tc_loss_ha"].fillna(0.0)
# primary loss is unavailable for 2001 -> keep NaN there (tree models handle NaN natively)
df = df.sort_values(KEY + ["year"]).reset_index(drop=True)

# ---- feature engineering ------------------------------------------------------------------------
df["total_loss_ha"] = df[DRIVERS].sum(axis=1)
audit["driver_sum_vs_tc_loss_ratio"] = float(df["total_loss_ha"].sum() / max(df["tc_loss_ha"].sum(), 1))
DENS, EXT = "avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1", "umd_tree_cover_extent_2000__ha"
df["agri_loss_ratio"] = (df["Permanent agriculture"] + df["Shifting cultivation"]) / (df["total_loss_ha"] + 1e-6)
df["wildfire_ratio"] = df["Wildfire"] / (df["total_loss_ha"] + 1e-6)
df["loss_extent_ratio"] = df["total_loss_ha"] / (df[EXT] + 1)
df["potential_co2e"] = df["total_loss_ha"] * df[DENS] * 44 / 12
# dynamic forest state (known BEFORE year t): cumulative earlier loss and remaining extent
g = df.groupby(KEY)
df["cum_loss_prior_ha"] = g["tc_loss_ha"].cumsum() - df["tc_loss_ha"]
df["remaining_extent_ha"] = (df[EXT] - df["cum_loss_prior_ha"]).clip(lower=0)
df["remaining_frac"] = df["remaining_extent_ha"] / (df[EXT] + 1)
df["loss_remaining_ratio"] = df["total_loss_ha"] / (df["remaining_extent_ha"] + 1)
df["remaining_carbon_Mg_C"] = df["remaining_extent_ha"] * df[DENS]
# primary forest: the high-carbon, hard-to-regrow share of loss
df["primary_share"] = df["primary_loss_ha"] / (df["tc_loss_ha"] + 1e-6)
df["forest_cover_frac"] = df[EXT] / (df["area_ha"] + 1)

# "hidden carbon" proxy (peat, belowground): the kabupaten's own emission per hectare lost in PRIOR years only.
# Prior-years-only => no leakage in temporal validation; unavailable (NaN) until >50 ha have been lost.
cum_e = g["gross_emissions_Mg_CO2e"].cumsum() - df["gross_emissions_Mg_CO2e"]
cum_l = g["total_loss_ha"].cumsum() - df["total_loss_ha"]
df["hist_intensity"] = np.where(cum_l > 50, cum_e / (cum_l + 1), np.nan)
df["hist_intensity_x_loss"] = df["hist_intensity"] * df["total_loss_ha"]

df = df.rename(columns={"gfw_forest_carbon_gross_removals__Mg_CO2_yr-1": "removals_avg_Mg_CO2_yr",
                        "gfw_forest_carbon_net_flux__Mg_CO2e_yr-1": "net_flux_avg_Mg_CO2e_yr"})
static = static.rename(columns={"gfw_forest_carbon_gross_removals__Mg_CO2_yr-1": "removals_avg_Mg_CO2_yr",
                                "gfw_forest_carbon_net_flux__Mg_CO2e_yr-1": "net_flux_avg_Mg_CO2e_yr",
                                "gfw_forest_carbon_gross_emissions__Mg_CO2e_yr-1": "gross_avg_Mg_CO2e_yr"})
static["net_status"] = np.where(static["net_flux_avg_Mg_CO2e_yr"] < 0, "Net sink", "Net source")

audit["national_gross_Mt_yr"] = float(static["gross_avg_Mg_CO2e_yr"].sum() / 1e6)
audit["national_removals_Mt_yr"] = float(static["removals_avg_Mg_CO2_yr"].sum() / 1e6)
audit["national_net_Mt_yr"] = float(static["net_flux_avg_Mg_CO2e_yr"].sum() / 1e6)
audit["n_net_sink"] = int((static["net_status"] == "Net sink").sum())

# leakage guard: net flux and annual-average gross are derived from the target -> not model features
df = df.drop(columns=["gross_avg_Mg_CO2e_yr"], errors="ignore")
df.to_csv("processed_forest_carbon.csv", index=False)
static.to_csv("kabupaten_static.csv", index=False)
json.dump(audit, open("data_audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2)); print(df.shape, df["subnational2"].nunique())

#### Apa yang didapat
| Indikator audit | Nilai |
|---|---|
| Kabupaten di sheet karbon / sheet pemicu | **497** / **494** |
| Baris panel | **12.425** (kolom: 39) |
| Baris tanpa catatan pemicu | **483** (sekitar 3,9% baris) |
| Rasio total pemicu terhadap total kehilangan tutupan | **0,998** |
| Emisi kotor nasional (rata-rata) | **943,5 Mt CO2e/tahun** |
| Serapan nasional | **594,4 Mt CO2e/tahun** |
| Net flux nasional | **+349,1 Mt CO2e/tahun** |
| Kabupaten *net sink* | **308 dari 497** |

Berkas keluaran: `processed_forest_carbon.csv`, `kabupaten_static.csv`, `data_audit.json`.

#### Penjelasan
* Rasio **0,998** berarti penjumlahan 7 pemicu hampir persis sama dengan total kehilangan tutupan, jadi **tabel pemicu konsisten** dengan tabel utama.
* Nasional, hutan Indonesia secara rata-rata adalah **net source** (+349 Mt), tetapi **62% kabupaten adalah net sink** (308/497). Artinya persoalan emisi **terkonsentrasi di sebagian wilayah**, bukan merata. Ini diperdalam di analisis Pareto (bagian 4.1).
* Panel memuat 497 pasangan provinsi-kabupaten tetapi hanya 496 nama `subnational2` unik, kemungkinan ada nama kabupaten yang sama di dua provinsi. Karena itu seluruh pengelompokan memakai **pasangan provinsi + kabupaten**, bukan nama kabupaten saja.
* Net flux GFW adalah **rata-rata 2001-2025** (bukan deret tahunan) dan bukan metodologi inventori FOLU resmi.

## 2. Eksplorasi Data (EDA)

**Ini bagian apa:** gambaran awal sebelum pemodelan: bagaimana emisi nasional bergerak dari tahun ke tahun, dan pemicu mana yang menyumbang kehilangan tutupan terbanyak.

**Penjelasan:** EDA memeriksa apakah data masuk akal dan apakah ada lonjakan yang perlu diwaspadai. Panel kiri menjumlahkan emisi seluruh kabupaten per tahun; panel kanan menjumlahkan luas kehilangan tutupan 2001-2025 per pemicu.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

nat = df.groupby("year")["gross_emissions_Mg_CO2e"].sum() / 1e6
ax[0].plot(nat.index, nat.values, marker="o", color="#2e7d32")
ax[0].set_title("Emisi kotor nasional per tahun"); ax[0].set_xlabel("Tahun"); ax[0].set_ylabel("Mt CO2e"); ax[0].grid(alpha=.3)

drv = df[DRIVERS].sum().sort_values() / 1e6
ax[1].barh(drv.index, drv.values, color="#8d6e63")
ax[1].set_title("Total kehilangan tutupan 2001-2025 per pemicu"); ax[1].set_xlabel("juta ha")

plt.tight_layout(); plt.show()
print(f"Tahun emisi nasional tertinggi: {int(nat.idxmax())} ({nat.max():,.0f} Mt CO2e)")
print(f"Pemicu dengan kehilangan terbesar: {drv.index[-1]} ({drv.iloc[-1]:,.2f} juta ha, {drv.iloc[-1]/drv.sum():.0%} dari total)")

#### Apa yang didapat
Dua grafik (deret emisi nasional dan komposisi kehilangan per pemicu) serta dua baris ringkasan: **tahun puncak emisi** dan **pemicu dominan**.

#### Penjelasan
* Periksa grafik kiri: bila deret emisi nasional tidak mulus dan melonjak pada tahun tertentu, itu alasan bagian 4.7 mendeteksi **anomali per kabupaten-tahun** (hasilnya terkumpul di 2015-2016) dan bagian 4.8 memakai *backtest* serta interval, bukan satu garis tren.
* Perbandingan pemicu menunjukkan komposisi **luas** kehilangan. Namun luas bukan emisi: satu hektar hutan primer berbiomassa tinggi melepas jauh lebih banyak karbon daripada satu hektar lahan sekunder. Itu sebabnya project ini menghitung **intensitas emisi per hektar** (bagian 3 dan 4.2).

## 3. Pemodelan & Validasi Tanpa Kebocoran

**Ini bagian apa:** *Modeling* + *Evaluation*. Beberapa model dibandingkan secara adil, lalu model terbaik dilatih final untuk dashboard.

**Penjelasan:**
* **Target:** emisi = luas hilang x intensitas; model memprediksi **log-intensitas** (lihat Bagian II.2).
* **Validasi:** (a) *future years* - latih <= 2021, uji 2022-2025; (b) *kabupaten tak terlihat* - `GroupKFold`. **Bukan** *random split*, karena itu membocorkan tahun-tahun kabupaten yang sama ke data uji.
* **Set fitur:** **COLD** (tanpa riwayat) dan **WARM** (+ `hist_intensity`; dipakai di aplikasi).
* **Pembanding:** Ridge (baseline linear), Random Forest, dan HistGradientBoosting dengan tiga bentuk target (mentah, log1p, intensitas).

### 3.1 Perbandingan model

In [ ]:
SEED = 42
KEY = ["subnational1", "subnational2"]
T = "gross_emissions_Mg_CO2e"
DENS = "avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1"
EXT = "umd_tree_cover_extent_2000__ha"
STK = "gfw_aboveground_carbon_stocks_2000__Mg_C"
DRIVERS = ["Hard commodities", "Logging", "Other natural disturbances", "Permanent agriculture",
           "Settlements & Infrastructure", "Shifting cultivation", "Wildfire"]
COLD = DRIVERS + ["total_loss_ha", "agri_loss_ratio", "wildfire_ratio", "loss_extent_ratio", "potential_co2e",
                  "primary_loss_ha", "primary_share", "remaining_extent_ha", "remaining_frac", "loss_remaining_ratio",
                  "remaining_carbon_Mg_C", "cum_loss_prior_ha", "area_ha", "forest_cover_frac",
                  EXT, STK, DENS, "year"]
WARM = COLD + ["hist_intensity", "hist_intensity_x_loss"]

df = pd.read_csv("processed_forest_carbon.csv")
LAST = int(df["year"].max())
y = df[T]
groups = df["subnational1"] + "|" + df["subnational2"]
train_t = df["year"] <= 2021


def hgb(**kw):
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEED, **kw)


LOSS = df["total_loss_ha"]


def fit_predict(model, Xa, ya, Xb, mode="raw"):
    """mode: 'raw' | 'log' (log1p target) | 'ratio' (emission INTENSITY = log((E+1)/(loss+1)), then x loss).
    The intensity form scales with hectares, so it does not saturate for large what-if inputs."""
    la, lb = LOSS.loc[Xa.index], LOSS.loc[Xb.index]
    if mode == "ratio":
        model.fit(Xa, np.log((ya + 1) / (la + 1)))
        return np.clip((lb + 1) * np.exp(model.predict(Xb)) - 1, 0, None)
    model.fit(Xa, np.log1p(ya) if mode == "log" else ya)
    p = model.predict(Xb)
    return np.clip(np.expm1(p) if mode == "log" else p, 0, None)


def evaluate(name, factory, feats, mode, cold_eval):
    X = df[feats].fillna(0) if factory().__class__.__name__ in ("Ridge", "RandomForestRegressor") else df[feats]
    p_t = fit_predict(factory(), X[train_t], y[train_t], X[~train_t], mode)
    row = {"Model": name, "R2 future years (2022-25)": r2_score(y[~train_t], p_t),
           "MAE future years": mean_absolute_error(y[~train_t], p_t)}
    if cold_eval:
        oof = np.zeros(len(y))
        for a, b in GroupKFold(5).split(X, y, groups):
            oof[b] = fit_predict(factory(), X.iloc[a], y.iloc[a], X.iloc[b], mode)
        row.update({"R2 unseen kabupaten": r2_score(y, oof), "MAE unseen kabupaten": mean_absolute_error(y, oof)})
    return row


rows = [
    evaluate("Ridge (cold features)", lambda: Ridge(alpha=1.0), COLD, "raw", True),
    evaluate("RandomForest raw target (cold)", lambda: RandomForestRegressor(150, min_samples_leaf=2, n_jobs=-1, random_state=SEED), COLD, "raw", True),
    evaluate("HistGB raw target (cold)", hgb, COLD, "raw", True),
    evaluate("HistGB log1p target (cold)", hgb, COLD, "log", True),
    evaluate("HistGB intensity target (cold)", hgb, COLD, "ratio", True),
    evaluate("HistGB intensity target (WARM, deployed)", hgb, WARM, "ratio", False),
]
res = pd.DataFrame(rows)
print(res.round(3).to_string())

# visual: MAE tahun uji (lebih kecil = lebih baik)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.barh(res["Model"][::-1], res["MAE future years"][::-1] / 1e3, color="#2e7d32")
ax.set_xlabel("MAE tahun uji 2022-2025 (ribu Mg CO2e per kabupaten-tahun)")
ax.set_title("Perbandingan model: MAE future years (lebih kecil = lebih baik)")
plt.tight_layout(); plt.show()

#### Apa yang didapat
Tabel metrik (satuan MAE: Mg CO2e per kabupaten-tahun) dan grafik batang MAE.

| Model | R² 2022-25 | MAE 2022-25 | R² kab. tak terlihat | MAE kab. tak terlihat |
|---|---|---|---|---|
| Ridge (baseline) | 0,917 | 343.294 | **0,974** | 327.798 |
| Random Forest, target mentah | 0,952 | 180.525 | 0,943 | 279.383 |
| HistGB, target mentah | 0,948 | 178.456 | 0,941 | 292.132 |
| HistGB, log1p | 0,966 | 165.517 | 0,922 | 285.811 |
| HistGB, **intensitas** (COLD) | 0,975 | 155.707 | 0,963 | **253.771** |
| **HistGB, intensitas (WARM, dipakai app)** | **0,977** | **140.562** | - | - |

*(Angka dari eksekusi pada data GFW; versi pustaka yang berbeda dapat menggeser angka sedikit. Tabel tepat Anda tercetak pada sel di atas.)*

#### Penjelasan
1. **Target intensitas terbukti membantu:** pada HistGB, MAE tahun uji turun dari 178.456 (target mentah) ke 155.707, sekitar **13% lebih baik**, dan MAE kabupaten tak terlihat ikut turun (292.132 -> 253.771).
2. **Terhadap baseline Ridge**, HistGB-intensitas menurunkan MAE sekitar **55%** pada tahun mendatang dan sekitar **23%** pada kabupaten tak terlihat.
3. **R² dan MAE bisa bertentangan:** Ridge punya R² kabupaten tak terlihat tertinggi (0,974) tetapi MAE terburuk. R² didominasi segelintir kabupaten raksasa; MAE lebih mewakili galat pada kabupaten biasa. **Kedua metrik ditampilkan** agar tidak menyesatkan.
4. **WARM vs COLD:** menambahkan riwayat intensitas kabupaten menurunkan MAE dari 155.707 menjadi 140.562 (sekitar 10%). Evaluasi WARM bersifat *one-step-ahead*: `hist_intensity` memakai emisi aktual tahun-tahun sebelumnya, jadi **tidak langsung sebanding** dengan COLD dan tidak berlaku untuk kabupaten tanpa riwayat.

### 3.2 Model final, interval ketidakpastian, dan fitur penting

**Ini bagian apa:** pelatihan model final (WARM) pada seluruh data, dua model kuantil untuk **interval 80%** (P10-P90), dan analisis fitur penting.

**Penjelasan:** *Permutation importance* diukur pada **tahun uji 2022-2025** (bukan data latih) dengan target log-intensitas, jadi menjawab pertanyaan "apa yang menjelaskan emisi **per hektar** yang hilang". Hasilnya dikelompokkan per konsep.

In [ ]:
# ---- final models (WARM) + 80% interval models ---------------------------------------------------
X = df[WARM]
ylog = np.log((y + 1) / (LOSS + 1))           # intensity target (deployed)
final = hgb().fit(X, ylog)
q10 = hgb(loss="quantile", quantile=0.10).fit(X, ylog)
q90 = hgb(loss="quantile", quantile=0.90).fit(X, ylog)

# ---- permutation importance on held-out years, grouped by concept ----------------------------------
m_eval = hgb().fit(X[train_t], ylog[train_t])
perm = permutation_importance(m_eval, X[~train_t], ylog[~train_t], n_repeats=8, random_state=SEED, scoring="r2")
# NOTE: importance is measured on the log-INTENSITY target, i.e. "what explains emission per hectare lost".
imp = pd.DataFrame({"Feature": WARM, "Importance": perm.importances_mean}).sort_values("Importance", ascending=False)
imp.to_csv("feature_importance.csv", index=False)
plt.figure(figsize=(10, 7))
top = imp.head(14)[::-1]
plt.barh(top["Feature"], top["Importance"], color="#2e7d32")
plt.title("Permutation importance for emission INTENSITY (held-out years 2022-2025)")
plt.xlabel("Drop in R² (log target)")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=200); plt.show()
print(imp.head(10).round(4).to_string())

#### Apa yang didapat
Grafik 14 fitur terpenting dan tabel 10 teratas (disimpan di `feature_importance.csv/.png`). Nilai teratas pada eksekusi:

| Fitur | Penurunan R² |
|---|---|
| `total_loss_ha` | **0,3405** |
| `remaining_extent_ha` | 0,0416 |
| `gfw_aboveground_carbon_stocks_2000__Mg_C` | 0,0223 |
| `hist_intensity` | 0,0180 |
| `potential_co2e` | 0,0121 |
| `area_ha` | 0,0118 |

#### Penjelasan
* **`total_loss_ha` mendominasi** (0,34), jauh di atas fitur lain. Bahkan pada target intensitas, skala kehilangan tetap paling berpengaruh; salah satu penjelasan yang masuk akal adalah intensitas kabupaten dengan kehilangan kecil lebih berfluktuasi, sehingga ukuran kehilangan ikut menentukan intensitasnya (ini hipotesis, belum diuji terpisah).
* Berikutnya muncul **sisa tutupan**, **stok karbon awal**, dan **riwayat intensitas kabupaten** (`hist_intensity`, proksi karbon tersembunyi/gambut), yaitu faktor biofisik wilayah, bukan sekadar jenis pemicu.
* Pernyataan lama bahwa emisi "sangat sensitif terhadap *wildfire ratio*" **tidak didukung data** dan sengaja dihapus. Importance ini bersifat **asosiatif**, bukan sebab-akibat.

### 3.3 Cek visual: prediksi vs aktual (tahun uji 2022-2025)

**Ini bagian apa:** diagnosis visual model yang dilatih hingga 2021 pada tahun-tahun yang belum pernah dilihat.

**Penjelasan:** setiap titik adalah satu kabupaten-tahun. Titik di sepanjang garis putus-putus berarti prediksi = aktual. Sumbu logaritmik dipakai karena emisi berbeda hingga beberapa orde besaran antarkabupaten.

In [ ]:
test = ~train_t
pred = np.clip((LOSS[test] + 1) * np.exp(m_eval.predict(X[test])) - 1, 0, None)
print(f"Cek WARM tahun uji: R2 = {r2_score(y[test], pred):.3f} | MAE = {mean_absolute_error(y[test], pred):,.0f}")

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(y[test] + 1, pred + 1, s=8, alpha=.4, color="#2e7d32")
lim = [1, max((y[test] + 1).max(), (pred + 1).max())]
ax.plot(lim, lim, "k--", lw=1, label="prediksi = aktual")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Emisi aktual (Mg CO2e + 1)"); ax.set_ylabel("Emisi prediksi (Mg CO2e + 1)")
ax.set_title("Prediksi vs aktual, tahun uji 2022-2025"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

#### Apa yang didapat
Satu baris metrik (harus konsisten dengan baris WARM pada tabel 3.1: R² sekitar 0,98 dan MAE sekitar 140 ribu) dan diagram sebar log-log.

#### Penjelasan
Titik yang menempel pada garis diagonal menunjukkan model mengikuti aktual di seluruh rentang skala. Sebaran yang lebih lebar pada emisi kecil (kiri-bawah) wajar: pada skala kecil, selisih kecil dalam Mg CO2e tampak besar di sumbu logaritmik padahal dampak absolutnya kecil. Ini sekaligus alasan MAE dibaca bersama R².

### 3.4 Status akhir kabupaten & penyimpanan model

**Ini bagian apa:** menyiapkan *state* setiap kabupaten pada akhir tahun terakhir (2025) sebagai input simulasi *what-if* tahun 2026 dan seterusnya, lalu menyimpan semua artefak untuk dashboard.

**Penjelasan:** `state` memuat kehilangan kumulatif, sisa tutupan, sisa karbon, dan riwayat intensitas tiap kabupaten. Aplikasi memakai ini ditambah masukan pengguna (luas kehilangan per pemicu) untuk memprediksi emisi skenario. Model disimpan bersama tiga keluaran: rata-rata, P10, dan P90.

In [ ]:
# ---- kabupaten state at end of LAST year (inputs for what-if simulation of LAST+1...) ---------------
g = df.sort_values(KEY + ["year"]).groupby(KEY)
cum_loss = g["tc_loss_ha"].sum()
cum_e, cum_l = g[T].sum(), g["total_loss_ha"].sum()
st = df.drop_duplicates(KEY)[KEY + ["area_ha", EXT, STK, DENS]].set_index(KEY)
st["cum_loss_prior_ha"] = cum_loss
st["remaining_extent_ha"] = (st[EXT] - st["cum_loss_prior_ha"]).clip(lower=0)
st["remaining_frac"] = st["remaining_extent_ha"] / (st[EXT] + 1)
st["remaining_carbon_Mg_C"] = st["remaining_extent_ha"] * st[DENS]
st["forest_cover_frac"] = st[EXT] / (st["area_ha"] + 1)
st["hist_intensity"] = np.where(cum_l > 50, cum_e / (cum_l + 1), np.nan)
state = st.reset_index()
lookup = state[KEY + [EXT, STK, DENS]]

joblib.dump({"mean": final, "q10": q10, "q90": q90}, "model_emission.joblib", compress=3)
joblib.dump({"feature_names": WARM, "cold_features": COLD, "driver_list": DRIVERS, "lookup_wilayah": lookup,
             "state": state, "last_year": LAST, "target": "log_intensity"}, "metadata_carbon.joblib")
res.to_json("metrics.json", orient="records", indent=2)
print(f"Tersimpan: model_emission.joblib, metadata_carbon.joblib, metrics.json | state: {len(state)} kabupaten, tahun terakhir {LAST}")

#### Apa yang didapat
Tiga berkas: `model_emission.joblib` (model rata-rata + P10 + P90), `metadata_carbon.joblib` (nama fitur, daftar pemicu, *state* kabupaten, tahun terakhir), dan `metrics.json` (tabel perbandingan model).

#### Penjelasan
Ketiganya adalah **jembatan antara notebook dan dashboard**: aplikasi Streamlit tidak melatih ulang apa pun, hanya memuat berkas-berkas ini. Bila versi scikit-learn di lingkungan aplikasi berbeda, jalankan ulang bagian 3.

## 4. Lapisan Data Mining: Pareto, NNLS, Net Flux, Hotspot, SHAP, Klaster, Anomali, Proyeksi

**Ini bagian apa:** delapan analisis yang menjawab pertanyaan *deskriptif* dan *prospektif* (bukan hanya prediksi). Setiap analisis menghasilkan tabel/grafik yang juga dipakai dashboard.

**Penjelasan:** sel pertama menyiapkan konstanta dan memuat ulang panel; sel-sel berikutnya menjalankan **satu analisis per sel**. Sebagian sel memakai hasil sel sebelumnya (mis. `nt` dari 4.3 dipakai klasterisasi di 4.6), jadi jalankan berurutan.

In [ ]:
SEED = 42
T = "gross_emissions_Mg_CO2e"
DENS = "avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1"
EXTENT = "umd_tree_cover_extent_2000__ha"
STOCK = "gfw_aboveground_carbon_stocks_2000__Mg_C"
D = ["Hard commodities", "Logging", "Other natural disturbances", "Permanent agriculture",
     "Settlements & Infrastructure", "Shifting cultivation", "Wildfire"]

df = pd.read_csv("processed_forest_carbon.csv")
df["kab"] = df["subnational1"] + " | " + df["subnational2"]
LAST = int(df["year"].max())
print(f"Panel dimuat: {df.shape[0]:,} baris, {df['kab'].nunique()} kabupaten, tahun {df['year'].min()}-{LAST}")

### 4.1 Konsentrasi emisi (Pareto)

**Ini bagian apa:** mengukur seberapa **terkonsentrasi** emisi kumulatif pada segelintir kabupaten.

**Penjelasan:** kabupaten diurutkan dari emisi kumulatif terbesar; dihitung kontribusi kumulatifnya. Kurva yang melengkung tajam berarti segelintir wilayah menentukan sebagian besar emisi, sehingga **intervensi bisa difokuskan**.

In [ ]:
# ============================================================ 1. Pareto concentration
tot = df.groupby(["subnational1", "subnational2"])[T].sum().sort_values(ascending=False).reset_index()
tot["share"] = tot[T] / tot[T].sum()
tot["cum_share"] = tot["share"].cumsum()
tot["rank"] = np.arange(1, len(tot) + 1)
tot.to_csv("pareto.csv", index=False)
n80 = int((tot["cum_share"] < 0.8).sum() + 1)
print(f"[Pareto] {n80} of {len(tot)} kabupaten produce 80% of cumulative emissions")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tot["rank"], tot["cum_share"] * 100, color="#2e7d32")
ax.axhline(80, ls="--", color="grey"); ax.axvline(n80, ls="--", color="grey")
ax.set_xlabel("Peringkat kabupaten (emisi kumulatif tertinggi -> terendah)"); ax.set_ylabel("Kontribusi kumulatif (%)")
ax.set_title(f"Kurva Pareto: {n80} kabupaten = 80% emisi kumulatif"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

tot.head(10).assign(share_pct=lambda d: (d["share"] * 100).round(2))[["rank", "subnational1", "subnational2", "share_pct"]]

#### Apa yang didapat
**86 dari 497 kabupaten (sekitar 17%) menghasilkan 80% emisi kumulatif.** Tabel di bawah grafik memuat 10 kabupaten penyumbang terbesar beserta persentase kontribusinya (`pareto.csv`).

#### Penjelasan
Konsentrasi ini sedikit lebih tajam dari aturan 80/20 klasik: emisi dominan di segelintir wilayah. Implikasinya praktis, **memprioritaskan puluhan kabupaten teratas** menjangkau sebagian besar emisi historis, jauh lebih efisien daripada kebijakan yang merata di 497 wilayah.

### 4.2 Intensitas emisi per pemicu (NNLS)

**Ini bagian apa:** menaksir **berapa Mg CO2e dilepas per hektar** untuk tiap pemicu, serta perbandingan hutan primer vs non-primer.

**Penjelasan:** *Non-negative least squares* memodelkan `emisi ~ sum(beta_d x hektar_d)` dengan beta >= 0 (tidak mungkin ada emisi negatif dari kehilangan hutan). Dijalankan untuk seluruh data, per pita kerapatan karbon (rendah/sedang/tinggi), dan untuk primer vs non-primer. **Indikatif saja:** pemicu saling berkorelasi, jadi koefisien nol berarti "tidak dapat dipisahkan", bukan "tidak berbahaya".

In [ ]:
# ============================================================ 2. Driver emission intensity (NNLS)
# Non-negative least squares: emissions ~ sum(beta_d * hectares_d). beta = Mg CO2e per ha.
# Indicative only: drivers are correlated, so zero coefficients mean "not separable", not "harmless".
rows = []
df["density_band"] = pd.qcut(df[DENS], 3, labels=["low", "mid", "high"])
for name, sub in [("All", df)] + [(f"Density {b}", s) for b, s in df.groupby("density_band", observed=True)]:
    beta, _ = nnls(sub[D].values.astype(float), sub[T].values)
    for d, b in zip(D, beta):
        rows.append({"segment": name, "driver": d, "Mg_CO2e_per_ha": b})
# primary forest vs everything else (primary loss exists from 2002)
sp = df.dropna(subset=["primary_loss_ha"])
A = np.column_stack([sp["primary_loss_ha"], (sp["tc_loss_ha"] - sp["primary_loss_ha"]).clip(lower=0)])
beta, _ = nnls(A.astype(float), sp[T].values)
for nm, b in zip(["Primary forest loss", "Non-primary loss"], beta):
    rows.append({"segment": "Primary vs non-primary", "driver": nm, "Mg_CO2e_per_ha": b})
pd.DataFrame(rows).to_csv("driver_intensity.csv", index=False)
print(f"[NNLS] primary forest {beta[0]:,.0f} vs non-primary {beta[1]:,.0f} Mg CO2e per ha")

di = pd.read_csv("driver_intensity.csv")
a = di[di["segment"] == "All"].sort_values("Mg_CO2e_per_ha")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(a["driver"], a["Mg_CO2e_per_ha"], color="#8d6e63")
ax.set_xlabel("Mg CO2e per hektar (NNLS, seluruh data)"); ax.set_title("Intensitas emisi per pemicu (indikatif)")
plt.tight_layout(); plt.show()

di.pivot(index="driver", columns="segment", values="Mg_CO2e_per_ha").round(0)

#### Apa yang didapat
* **Hutan primer: 1.170 Mg CO2e/ha** vs **non-primer: 629 Mg CO2e/ha**.
* Grafik intensitas per pemicu dan tabel perbandingan lintas segmen kerapatan karbon (`driver_intensity.csv`).

#### Penjelasan
* Satu hektar hutan primer yang hilang melepas sekitar **1,9x** lebih banyak karbon daripada satu hektar non-primer. Hutan primer menyimpan biomassa tinggi dan sulit pulih, sehingga **melindungi hutan primer memberi manfaat karbon per hektar terbesar**.
* Baca koefisien per pemicu dengan hati-hati. Pemicu yang berkorelasi (mis. pembalakan dan pertanian pada lahan yang sama) membuat NNLS tidak selalu bisa memisahkan kontribusinya; koefisien nol bukan berarti pemicu itu tidak berbahaya.

### 4.3 Net flux: emisi dikurangi serapan

**Ini bagian apa:** membandingkan **emisi kotor** dengan **serapan** untuk menentukan kabupaten *net source* atau *net sink*.

**Penjelasan:** emisi kotor saja menyesatkan karena hutan yang tersisa juga menyerap karbon. Net flux dari data GFW (rata-rata 2001-2025) = emisi - serapan; nilai negatif berarti *net sink*. Ini dasar relevansi dengan target **FOLU Net Sink**.

In [ ]:
# ============================================================ 2b. Net flux (emissions minus removals) from raw data
stat = pd.read_csv("kabupaten_static.csv")
cc = df.groupby(["subnational1", "subnational2"]).agg(cum_E=(T, "sum"), cum_L=("total_loss_ha", "sum")).reset_index()
nt = stat.merge(cc, on=["subnational1", "subnational2"], how="left")
nt["net_per_ha"] = nt["net_flux_avg_Mg_CO2e_yr"] / (nt["extent_2000_ha"] + 1)
nt.to_csv("net_table.csv", index=False)
print(f"[Net] {int((nt['net_status']=='Net sink').sum())}/{len(nt)} kabupaten are net sinks; national net = "
      f"{nt['net_flux_avg_Mg_CO2e_yr'].sum()/1e6:,.0f} Mt CO2e/yr (gross {nt['gross_avg_Mg_CO2e_yr'].sum()/1e6:,.0f}, removals {nt['removals_avg_Mg_CO2_yr'].sum()/1e6:,.0f})")

fig, ax = plt.subplots(figsize=(6.5, 4))
vals = {"Emisi kotor": nt["gross_avg_Mg_CO2e_yr"].sum() / 1e6,
        "Serapan": -nt["removals_avg_Mg_CO2_yr"].sum() / 1e6,
        "Net flux": nt["net_flux_avg_Mg_CO2e_yr"].sum() / 1e6}
ax.bar(vals.keys(), vals.values(), color=["#c62828", "#2e7d32", "#455a64"])
ax.axhline(0, color="black", lw=.8); ax.set_ylabel("Mt CO2e per tahun"); ax.set_title("Neraca karbon hutan nasional (rata-rata 2001-2025)")
plt.tight_layout(); plt.show()

nt.nlargest(10, "net_flux_avg_Mg_CO2e_yr")[["subnational1", "subnational2", "net_flux_avg_Mg_CO2e_yr", "net_status"]]

#### Apa yang didapat
**308 dari 497 kabupaten adalah *net sink*.** Nasional: emisi kotor **943 Mt**, serapan **594 Mt**, net flux **+349 Mt CO2e/tahun** (net *source*). Tabel menampilkan 10 kabupaten *net source* terbesar (`net_table.csv`).

#### Penjelasan
Secara nasional hutan Indonesia masih menjadi sumber bersih, tetapi **mayoritas wilayah menyerap lebih dari yang dilepas**; selisih nasional kemungkinan besar ditentukan oleh segelintir kabupaten emitor besar (periksa tabel 10 teratas), sejalan dengan konsentrasi emisi pada analisis Pareto. Karena net flux adalah **rata-rata periode**, angka ini bukan inventori tahunan resmi FOLU.

### 4.4 Hotspot "karbon tersembunyi"

**Ini bagian apa:** mencari kabupaten yang emisinya **per hektar hilang jauh lebih besar** dari yang diharapkan dari biomassa di atas tanah.

**Penjelasan:** untuk kabupaten dengan kehilangan kumulatif >= 2.000 ha dihitung `emisi per ha hilang` dan dibagi ekspektasi biomassa atas tanah (`kerapatan x 44/12`). Rasio jauh di atas 1 menandakan ada karbon lain (mis. **gambut** atau biomassa bawah tanah) yang ikut terlepas. Data gambut belum dimasukkan, jadi ini **proksi**, bukan pembuktian.

In [ ]:
# ============================================================ 2c. Hidden-carbon hotspots (emission per ha lost vs peers)
hs = cc[cc["cum_L"] >= 2000].copy()
hs["Mg_CO2e_per_ha_lost"] = hs["cum_E"] / hs["cum_L"]
hs = hs.merge(stat[["subnational1", "subnational2", "avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1"]], on=["subnational1", "subnational2"])
hs["density_aboveground"] = hs["avg_gfw_aboveground_carbon_stocks_2000__Mg_C_ha-1"]
hs["intensity_vs_aboveground_expectation"] = hs["Mg_CO2e_per_ha_lost"] / (hs["density_aboveground"] * 44 / 12 + 1)
hs_out = hs.sort_values("intensity_vs_aboveground_expectation", ascending=False)[
    ["subnational1", "subnational2", "cum_L", "Mg_CO2e_per_ha_lost", "density_aboveground", "intensity_vs_aboveground_expectation"]
]
hs_out.to_csv("intensity_hotspots.csv", index=False)
print(f"[Hotspot] {len(hs_out)} kabupaten dianalisis (kehilangan kumulatif >= 2.000 ha)")
hs_out.head(10).round(2)

#### Apa yang didapat
Tabel 10 kabupaten dengan rasio intensitas tertinggi (`intensity_hotspots.csv`), berurutan dari yang paling di atas ekspektasi biomassa atas tanah.

#### Penjelasan
Kabupaten teratas adalah **kandidat wilayah berkarbon tersembunyi**. Nilai rasio > 1 berarti emisi per hektar melebihi yang bisa dijelaskan oleh biomassa di atas tanah. Wilayah ini layak diprioritaskan untuk verifikasi dengan **peta gambut** sebelum dijadikan dasar kebijakan, karena rasio tinggi juga dapat berasal dari faktor lain.

### 4.5 Model interpretasi dan SHAP

**Ini bagian apa:** menjelaskan **pemicu mana yang paling mendorong prediksi emisi**.

**Penjelasan:** dipakai model terpisah **tanpa fitur agregat** (`total_loss_ha`, `potential_co2e`) agar kredit tidak "bersembunyi" di dalam fitur jumlah-semua-pemicu. Model ini hanya untuk **penjelasan**; aplikasi memprediksi dengan model utama. SHAP menghitung kontribusi tiap fitur terhadap prediksi; nilai `mean |SHAP|` adalah ukuran pengaruh rata-rata.

In [ ]:
# ============================================================ 3. Interpretation model + SHAP
# A separate model WITHOUT aggregate features (total_loss/potential_co2e), so credit is not hidden
# inside the sum-of-drivers feature. Used for explanation only; the app predicts with the main model.
FI = D + [EXTENT, DENS, "primary_share", "remaining_frac"]
Xi, yi = df[FI], np.log1p(df[T])
oof = np.zeros(len(yi))
for a, b in GroupKFold(5).split(Xi, yi, df["kab"]):
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEED).fit(Xi.iloc[a], yi.iloc[a])
    oof[b] = m.predict(Xi.iloc[b])
r2_interp = r2_score(yi, oof)
mi = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEED).fit(Xi, yi)
samp = Xi.sample(2000, random_state=SEED)
sv = shap.TreeExplainer(mi).shap_values(samp)
shap_df = pd.DataFrame({"feature": FI, "mean_abs_shap": np.abs(sv).mean(0)}).sort_values("mean_abs_shap", ascending=False)
shap_df["interp_model_R2_unseen_kab"] = r2_interp
shap_df.to_csv("shap_driver.csv", index=False)
plt.figure(figsize=(9, 5))
shap.summary_plot(sv, samp, show=False, plot_size=None)
plt.title("SHAP (log-emisi): pengaruh tiap pemicu & fitur biofisik")
plt.tight_layout(); plt.savefig("shap_driver.png", dpi=170); plt.show()
print(f"[SHAP] interpretation model R2 (unseen kabupaten, log scale) = {r2_interp:.3f}")
shap_df.head(8).round(3)

#### Apa yang didapat
* Model interpretasi mencapai **R² = 0,962** (skala log, kabupaten tak terlihat), jadi cukup andal untuk dijadikan dasar penjelasan.
* *SHAP summary plot* (sumbu x: arah dan besar pengaruh; warna: nilai fitur) dan tabel `mean |SHAP|` 8 fitur teratas (`shap_driver.csv/.png`).

#### Penjelasan
* Fitur dengan `mean |SHAP|` terbesar adalah yang paling menggeser prediksi emisi. Titik merah (nilai fitur tinggi) di sisi kanan berarti nilai tinggi **menaikkan** prediksi emisi.
* SHAP menjelaskan **perilaku model**, bukan sebab-akibat di lapangan. Pemicu yang berkorelasi dapat saling mengambil kredit.

### 4.6 Klasterisasi: tipologi kabupaten

**Ini bagian apa:** mengelompokkan kabupaten dengan profil emisi serupa agar strategi dapat **disesuaikan per tipe**.

**Penjelasan:** setiap kabupaten diringkas oleh komposisi pemicu (porsi tiap pemicu), emisi total (log), kehilangan rerata, kerapatan karbon, rasio kehilangan terhadap tutupan, emisi per ha, porsi hutan primer, net flux per ha, status net *source*, dan **tren relatif emisi**. Fitur distandarkan, jumlah klaster **K dipilih dengan silhouette** (K=3..7), lalu PCA 2 dimensi untuk visualisasi. Setiap klaster diberi nama otomatis dari pemicu dominannya.

In [ ]:
# ============================================================ 4. Clustering: kabupaten typology
g = df.groupby(["subnational1", "subnational2"])
prof = g[D].sum()
share = prof.div(prof.sum(axis=1) + 1e-9, axis=0).add_prefix("share_")
agg = pd.DataFrame({
    "log_total_emission": np.log1p(g[T].sum()),
    "log_mean_loss": np.log1p(g["total_loss_ha"].mean()),
    "density": g[DENS].first(),
    "loss_extent_ratio": g["loss_extent_ratio"].mean(),
    "emission_per_ha": g[T].sum() / (g["total_loss_ha"].sum() + 1),
    "primary_share": g["primary_share"].mean(),
})
_nt = nt.set_index(["subnational1", "subnational2"])
agg["net_flux_per_ha"] = (_nt["net_flux_avg_Mg_CO2e_yr"] / (_nt["extent_2000_ha"] + 1)).reindex(agg.index).clip(-100, 100)
agg["net_source"] = (_nt["net_status"] == "Net source").reindex(agg.index).astype(float)
def slope(s):
    return np.polyfit(s["year"], s[T], 1)[0] / (s[T].mean() + 1) if len(s) > 4 else 0.0
agg["trend_rel"] = df.groupby(["subnational1", "subnational2"]).apply(slope, include_groups=False).clip(-0.3, 0.3)
Z = pd.concat([share, agg], axis=1).fillna(0)
Zs = StandardScaler().fit_transform(Z)
sil = {k: silhouette_score(Zs, KMeans(k, n_init=10, random_state=SEED).fit_predict(Zs)) for k in range(3, 8)}
K = max(sil, key=sil.get)
km = KMeans(K, n_init=20, random_state=SEED).fit(Zs)
Z["cluster"] = km.labels_
pc = PCA(2, random_state=SEED).fit_transform(Zs)
Z["pc1"], Z["pc2"] = pc[:, 0], pc[:, 1]

LAB = {"Permanent agriculture": "Agrikultur/Sawit", "Shifting cultivation": "Ladang Berpindah",
       "Logging": "Pembalakan", "Wildfire": "Kebakaran", "Hard commodities": "Tambang",
       "Settlements & Infrastructure": "Infrastruktur", "Other natural disturbances": "Gangguan Alami"}
summ = Z.groupby("cluster").agg(n=("density", "size"), density=("density", "mean"),
                                mean_total_emission=("log_total_emission", lambda s: np.expm1(s).mean()),
                                emission_per_ha=("emission_per_ha", "mean"), trend=("trend_rel", "mean"),
                                primary_share=("primary_share", "mean"), net_source_share=("net_source", "mean"),
                                **{c: (c, "mean") for c in share.columns}).reset_index()
names = {}
em_median = summ["mean_total_emission"].median()
for _, r in summ.iterrows():
    tops = [d for d in sorted(D, key=lambda d: -r[f"share_{d}"]) if r[f"share_{d}"] >= 0.15][:3] \
        or [max(D, key=lambda d: r[f"share_{d}"])]
    level = "emisi tinggi" if r["mean_total_emission"] >= em_median else "emisi rendah"
    names[int(r["cluster"])] = f"C{int(r['cluster'])+1}: " + " + ".join(LAB[d] for d in tops) + f" ({level})"
summ["label"] = summ["cluster"].map(names)
Z["cluster_label"] = Z["cluster"].map(names)
Z.reset_index().to_csv("cluster_table.csv", index=False)
summ.to_csv("cluster_summary.csv", index=False)
print(f"[Cluster] best K={K} (silhouette={sil[K]:.3f}); all: { {k: round(v,3) for k,v in sil.items()} }")

fig, ax = plt.subplots(figsize=(9, 6))
for lab, s in Z.groupby("cluster_label"):
    ax.scatter(s["pc1"], s["pc2"], s=18, alpha=.7, label=f"{lab} (n={len(s)})")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("Tipologi kabupaten (PCA 2D dari profil emisi)")
ax.legend(fontsize=8, loc="best"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

summ[["label", "n", "density", "mean_total_emission", "emission_per_ha", "trend", "net_source_share"]].round(2)

#### Apa yang didapat
* **K terbaik = 5** (silhouette = **0,257**; K=3: 0,241, K=4: 0,255, K=6: 0,242, K=7: 0,222).
* Peta PCA 2D berwarna per klaster dan tabel ringkasan klaster berisi jumlah anggota, kerapatan karbon, emisi total rerata, emisi per ha, tren, dan porsi net *source* (`cluster_table.csv`, `cluster_summary.csv`).

#### Penjelasan
* Nilai silhouette **0,257** menunjukkan struktur klaster **sedang, bukan terpisah tegas**: antar-klaster ada tumpang tindih dan selisih silhouette antar-K kecil. Klaster ini adalah **alat deskriptif untuk menyusun tipologi dan strategi**, bukan batas alamiah yang pasti.
* Nama klaster otomatis menyebut pemicu dominan (porsi >= 15%) dan level emisi (tinggi/rendah relatif median klaster). Contoh pemakaian: klaster dominan **kebakaran** ditangani dengan pencegahan karhutla; dominan **agrikultur/sawit** dengan tata kelola izin dan lahan.

### 4.7 Deteksi anomali (tahun-tahun kejadian)

**Ini bagian apa:** menemukan **kabupaten-tahun** ketika emisi melonjak jauh di atas kebiasaan kabupaten itu sendiri.

**Penjelasan:** dipakai *robust z-score* berbasis **median dan MAD** (bukan rata-rata/standar deviasi) pada log-emisi per kabupaten, sehingga tahan terhadap pencilan. Ditandai anomali bila `robust z > 3` dan emisi > 300 ribu Mg CO2e (menyaring lonjakan yang tidak material). Penyebab diduga: **kebakaran** bila porsi *wildfire* > 40%, selain itu pemicu terbesar pada tahun tersebut.

In [ ]:
# ============================================================ 5. Anomaly detection (event years)
df = df.sort_values(["kab", "year"])
df["lg"] = np.log1p(df[T])
med = df.groupby("kab")["lg"].transform("median")
mad = df.groupby("kab")["lg"].transform(lambda s: (s - s.median()).abs().median()) + 0.05
df["robust_z"] = 0.6745 * (df["lg"] - med) / mad
an = df[(df["robust_z"] > 3.0) & (df[T] > 3e5)].copy()
an["wildfire_share"] = an["Wildfire"] / (an["total_loss_ha"] + 1e-6)
an["likely_cause"] = np.where(an["wildfire_share"] > 0.4, "Kebakaran",
                      an[D].idxmax(axis=1).map(LAB))
an_out = an[["subnational1", "subnational2", "year", T, "total_loss_ha", "robust_z", "wildfire_share", "likely_cause"]] \
    .sort_values(T, ascending=False)
an_out.to_csv("anomalies.csv", index=False)
print(f"[Anomaly] {len(an)} kabupaten-years flagged; top years: {an['year'].value_counts().head(4).to_dict()}")

if len(an):
    fig, ax = plt.subplots(figsize=(8, 3.8))
    cnt = an["year"].value_counts().sort_index()
    ax.bar(cnt.index.astype(str), cnt.values, color="#c62828")
    ax.set_xlabel("Tahun"); ax.set_ylabel("Jumlah kabupaten beranomali"); ax.set_title("Tahun kejadian lonjakan emisi")
    plt.tight_layout(); plt.show()

an_out.head(10).round(2)

#### Apa yang didapat
**20 kabupaten-tahun** ditandai anomali. Tahun terbanyak: **2015 (6)** dan **2016 (5)**, diikuti 2007 (2) dan 2009 (1). Tabel memuat 10 kejadian terbesar beserta dugaan penyebab (`anomalies.csv`).

#### Penjelasan
Pengelompokan pada 2015-2016 **konsisten dengan** musim kebakaran hebat tahun-tahun tersebut, dan kolom `likely_cause` menunjukkan seberapa banyak anomali berasal dari kebakaran. Penyebab di sini adalah **dugaan berbasis porsi pemicu**, bukan konfirmasi; kolom ini membantu menentukan kasus mana yang layak diselidiki lebih lanjut.

### 4.8 Proyeksi 5 tahun (2026-2030) dengan *backtest*

**Ini bagian apa:** memproyeksikan emisi tiap kabupaten untuk 1 sampai 5 tahun ke depan, **menguji apakah proyeksi itu benar-benar lebih baik daripada tebakan sederhana**, dan memberi interval ketidakpastian.

**Penjelasan:**
* **Direct multi-horizon:** satu model terpisah untuk setiap horizon h = 1..5 (bukan rekursif, sehingga galat tidak menumpuk).
* ***Rolling-origin backtest*** pada origin 2012, 2015, 2018, 2020: model hanya dilatih dengan informasi yang tersedia pada origin, lalu diuji ke h tahun berikutnya. Pembanding: ***naive* tahun lalu** dan ***naive* rata-rata 3 tahun**.
* **Pemilihan metode per horizon** berdasarkan MAE terendah.
* **Interval 80%** dari kuantil 10% dan 90% galat log hasil *backtest* metode terpilih.

In [ ]:
# ============================================================ 6. Multi-year forecast (direct, h=1..5) + backtest
df["loss_roll3"] = df.groupby("kab")["total_loss_ha"].transform(lambda s: s.rolling(3, min_periods=1).mean())
df["em_roll3"] = df.groupby("kab")[T].transform(lambda s: s.rolling(3, min_periods=1).mean())
df["em_roll5"] = df.groupby("kab")[T].transform(lambda s: s.rolling(5, min_periods=1).mean())
df["em_lag1"] = df.groupby("kab")[T].shift(1)
df["em_lag2"] = df.groupby("kab")[T].shift(2)
df["fire_roll3"] = df.groupby("kab")["Wildfire"].transform(lambda s: s.rolling(3, min_periods=1).mean())
FF = ["year", T, "em_lag1", "em_lag2", "em_roll3", "em_roll5", "total_loss_ha", "loss_roll3", "fire_roll3",
      "Permanent agriculture", "Wildfire", "Logging", EXTENT, DENS, "hist_intensity", "primary_loss_ha", "remaining_frac"]
H = [1, 2, 3, 4, 5]
full = df.set_index(["kab", "year"])[T]

def target_h(frame, h):
    idx = pd.MultiIndex.from_arrays([frame["kab"], frame["year"] + h])
    return full.reindex(idx).values

def run_origin(origin):
    """Train only on information available at `origin`, forecast origin+1..origin+5."""
    out = []
    for h in H:
        hist = df[df["year"] + h <= origin]
        y = target_h(hist, h); ok = ~np.isnan(y)
        m = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.06, random_state=SEED)
        m.fit(hist.loc[ok, FF], np.log1p(y[ok]))
        cur = df[df["year"] == origin]
        pred = np.expm1(m.predict(cur[FF])).clip(0)
        act = target_h(cur, h)
        ok2 = ~np.isnan(act)
        naive1 = cur[T].values
        naive3 = cur["em_roll3"].values
        for name, p in [("HistGB-lag", pred), ("Naive: last year", naive1), ("Naive: 3-yr mean", naive3)]:
            out.append({"origin": origin, "h": h, "model": name,
                        "MAE": mean_absolute_error(act[ok2], p[ok2]),
                        "R2": r2_score(act[ok2], p[ok2]) if ok2.sum() > 2 else np.nan})
    return out

bt = pd.DataFrame(sum([run_origin(o) for o in (2012, 2015, 2018, 2020)], []))
bt_sum = bt.groupby(["model", "h"])[["MAE", "R2"]].mean().reset_index()
bt_sum.to_csv("forecast_backtest.csv", index=False)
print("[Forecast backtest] (mean over origins 2012/2015/2018/2020)")
print(bt_sum.pivot(index="h", columns="model", values="MAE").round(0).to_string())

# choose per-horizon best model on MAE; final forecast from LAST year
best = bt_sum.loc[bt_sum.groupby("h")["MAE"].idxmin()].set_index("h")["model"].to_dict()
cur = df[df["year"] == LAST].copy()
res_rows = []
resid_q = {}
for h in H:
    hist = df[df["year"] + h <= LAST]
    y = target_h(hist, h); ok = ~np.isnan(y)
    m = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.06, random_state=SEED).fit(hist.loc[ok, FF], np.log1p(y[ok]))
    model_pred = np.expm1(m.predict(cur[FF])).clip(0)
    base = {"HistGB-lag": model_pred, "Naive: last year": cur[T].values, "Naive: 3-yr mean": cur["em_roll3"].values}[best[h]]
    # empirical 80% interval from backtest log-residuals of the chosen method (pooled across origins)
    lr = []
    for o in (2012, 2015, 2018, 2020):
        hh = df[df["year"] + h <= o]; yy = target_h(hh, h); k2 = ~np.isnan(yy)
        mm = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.06, random_state=SEED).fit(hh.loc[k2, FF], np.log1p(yy[k2]))
        c = df[df["year"] == o]; a = target_h(c, h); k3 = ~np.isnan(a)
        pr = {"HistGB-lag": np.expm1(mm.predict(c[FF])).clip(0), "Naive: last year": c[T].values, "Naive: 3-yr mean": c["em_roll3"].values}[best[h]]
        lr.extend((np.log1p(a[k3]) - np.log1p(pr[k3])).tolist())
    q10, q90 = np.quantile(lr, [0.1, 0.9])
    for i, (_, r) in enumerate(cur.iterrows()):
        res_rows.append({"subnational1": r["subnational1"], "subnational2": r["subnational2"], "year": LAST + h,
                         "forecast_Mg_CO2e": base[i],
                         "p10": np.expm1(np.log1p(base[i]) + q10), "p90": np.expm1(np.log1p(base[i]) + q90),
                         "method": best[h]})
fc = pd.DataFrame(res_rows)
fc.to_csv("forecast_2026_2030.csv", index=False)
print("[Forecast] chosen method per horizon:", best)
print(fc.groupby("year")["forecast_Mg_CO2e"].sum().round(-5).to_dict())

# visual: riwayat nasional + proyeksi titik
hist_nat = df.groupby("year")[T].sum() / 1e6
fc_nat = fc.groupby("year")["forecast_Mg_CO2e"].sum() / 1e6
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(hist_nat.index, hist_nat.values, marker="o", color="#2e7d32", label="Riwayat (aktual)")
ax.plot([hist_nat.index[-1]] + list(fc_nat.index), [hist_nat.values[-1]] + list(fc_nat.values),
        marker="s", ls="--", color="#c62828", label="Proyeksi (jumlah titik per kabupaten)")
ax.set_xlabel("Tahun"); ax.set_ylabel("Mt CO2e"); ax.set_title("Emisi kotor nasional: riwayat dan proyeksi"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

fc[fc["year"] == LAST + 1].nlargest(10, "forecast_Mg_CO2e")[["subnational1", "subnational2", "year", "forecast_Mg_CO2e", "p10", "p90", "method"]].round(0)

#### Apa yang didapat
**MAE *backtest* (rata-rata 4 origin; Mg CO2e per kabupaten-tahun):**

| Horizon | HistGB-lag | Naive tahun lalu | Naive rata-rata 3 th | Metode terpilih |
|---|---|---|---|---|
| h=1 | **1.061.671** | 1.127.628 | 1.126.261 | HistGB-lag |
| h=2 | **729.075** | 919.966 | 948.756 | HistGB-lag |
| h=3 | **1.027.841** | 1.151.516 | 1.143.748 | HistGB-lag |
| h=4 | **1.058.558** | 1.167.083 | 1.297.061 | HistGB-lag |
| h=5 | 1.072.517 | 1.211.957 | **1.057.873** | Naive 3 th |

**Proyeksi nasional (jumlah seluruh kabupaten, pembulatan 100 ribu Mg):** 2026: ~708 Mt, 2027: ~736 Mt, 2028: ~774 Mt, 2029: ~654 Mt, 2030: ~743 Mt CO2e. Berkas `forecast_backtest.csv` dan `forecast_2026_2030.csv` (dengan P10/P90 per kabupaten).

#### Penjelasan
* **Model unggul pada h = 1 sampai 4:** MAE lebih rendah sekitar 6% (h=1), 21% (h=2), 10% (h=3), dan 9% (h=4) dibanding *naive* terbaik. Pada **h=5 selisihnya kecil dan *naive* 3 tahun sedikit lebih baik**, sehingga metode itulah yang dipilih. Jadi sistem **jujur memakai tebakan sederhana saat model tidak lebih baik**.
* Total nasional **naik-turun antar-tahun** karena tiap horizon dimodelkan terpisah dan h=5 memakai metode berbeda. Jangan membacanya sebagai **tren** (mis. penurunan 2029 bukan prediksi kebijakan), melainkan sebagai estimasi independen per tahun.
* Interval P10-P90 dihitung **per kabupaten**; jumlah kuantil antar-kabupaten bukan interval nasional yang valid, sehingga grafik nasional hanya menampilkan titik.
* Pemilihan metode per horizon memakai *backtest* yang sama dengan kalibrasi interval, jadi akurasinya sedikit **optimistis**; hanya ada 4 origin, sehingga hasil perlu dibaca sebagai indikasi, bukan jaminan.

## 5. Deployment: Dashboard Streamlit

**Ini bagian apa:** *Deployment* - prototipe ***decision support system*** untuk simulasi skenario.

**Penjelasan:** dashboard ada di berkas terpisah `app.py`. Aplikasi **tidak melatih ulang model**; ia hanya memuat artefak yang dihasilkan notebook ini.

**Cara menjalankan:**
1. Jalankan seluruh sel notebook (*Run All*) hingga selesai.
2. Pastikan berkas berikut berada satu folder dengan `app.py`: `model_emission.joblib`, `metadata_carbon.joblib`, `processed_forest_carbon.csv`, `indonesia_provinces.geojson`, serta tabel hasil data mining (`pareto.csv`, `driver_intensity.csv`, `net_table.csv`, `intensity_hotspots.csv`, `shap_driver.csv`, `cluster_table.csv`, `cluster_summary.csv`, `anomalies.csv`, `forecast_2026_2030.csv`).
3. Di terminal: `streamlit run app.py`
4. Jika model gagal dimuat karena beda versi scikit-learn, jalankan ulang bagian 3 (atau `python train_improved.py`).

| Modul dashboard | Sumber data |
|---|---|
| Simulasi *what-if* emisi + interval P10/P90 | `model_emission.joblib` + `metadata_carbon.joblib` |
| Peta & peringkat konsentrasi emisi | `pareto.csv`, `net_table.csv` |
| Tipologi & anomali wilayah | `cluster_table.csv`, `anomalies.csv` |
| Proyeksi 2026-2030 | `forecast_2026_2030.csv` |

## 6. Kesimpulan, Rekomendasi, Keterbatasan, dan Pengembangan Lanjutan

### 6.1 Kesimpulan
1. **Estimasi emisi akurat dan tidak bocor.** Model intensitas (HistGB) menurunkan MAE sekitar **55%** (tahun mendatang) dan **23%** (kabupaten tak terlihat) dibanding Ridge, dengan R² 0,98 pada 2022-2025 untuk model WARM.
2. **Emisi sangat terkonsentrasi:** **86 dari 497 kabupaten** menyumbang 80% emisi kumulatif, sementara **308 kabupaten adalah *net sink***.
3. **Hutan primer bernilai karbon tinggi:** melepas sekitar **1,9x** lebih banyak CO2e per hektar dibanding non-primer (indikatif).
4. **Tipologi dan kejadian ekstrem dapat dipetakan:** 5 tipe kabupaten (struktur sedang) dan 20 kabupaten-tahun anomali, terbanyak pada 2015-2016.
5. **Proyeksi 2026-2030 lebih baik daripada tebakan naif** pada h=1 sampai 4, dengan interval ketidakpastian; pada h=5 sistem memilih metode naif karena lebih baik.

### 6.2 Rekomendasi Pemanfaatan
| Pemangku kepentingan | Pemanfaatan hasil |
|---|---|
| Pemerintah pusat | Prioritaskan intervensi pada kabupaten teratas hasil Pareto dan *net source* terbesar |
| Pemerintah daerah | Gunakan tipologi klaster untuk memilih strategi (pencegahan karhutla, tata kelola izin sawit, dsb.) |
| Perencana kebijakan | Gunakan simulator *what-if* untuk menguji skenario pengurangan kehilangan tutupan |
| Verifikator karbon | Verifikasi kabupaten hotspot dengan peta gambut sebelum dijadikan dasar |

### 6.3 Keterbatasan (disampaikan terbuka)
* Model memprediksi **emisi kotor tahunan**; **net flux hanya rata-rata 2001-2025** dan bukan metodologi inventori FOLU resmi.
* Hubungan pemicu-emisi bersifat **korelatif, bukan kausal** (importance, SHAP, dan NNLS).
* **Data gambut belum dimasukkan;** `hist_intensity` hanyalah proksi karbon tersembunyi.
* Evaluasi WARM bersifat *one-step-ahead* dan tidak berlaku untuk kabupaten tanpa riwayat (gunakan COLD).
* Struktur klaster **sedang** (silhouette 0,257); klaster adalah alat deskriptif.
* Proyeksi per horizon dimodelkan independen sehingga lintasan nasional tidak mulus; pemilihan metode dan kalibrasi interval memakai *backtest* yang sama (sedikit optimistis); hanya 4 origin.
* Harga karbon pada simulator adalah **asumsi pengguna**.

### 6.4 Pengembangan Lanjutan
* Menambahkan **peta gambut** dan data iklim (El Niño, curah hujan) sebagai fitur.
* Menambahkan **kuantifikasi ketidakpastian nasional** (agregasi interval dengan korelasi antar-kabupaten).
* Memvalidasi tipologi klaster dengan data lapangan atau kebijakan daerah.
* Menguji metode pembelajaran lain (mis. model deret waktu panel) pada proyeksi h=5.

## 7. Lampiran

### 7.1 Artefak keluaran
Sel berikut memeriksa berkas keluaran yang dihasilkan notebook sehingga mudah dicek sebelum menjalankan dashboard.

In [ ]:
artefak = ["processed_forest_carbon.csv", "kabupaten_static.csv", "data_audit.json",
           "model_emission.joblib", "metadata_carbon.joblib", "metrics.json",
           "feature_importance.csv", "feature_importance.png",
           "pareto.csv", "driver_intensity.csv", "net_table.csv", "intensity_hotspots.csv",
           "shap_driver.csv", "shap_driver.png", "cluster_table.csv", "cluster_summary.csv",
           "anomalies.csv", "forecast_backtest.csv", "forecast_2026_2030.csv"]
ceks = pd.DataFrame({"berkas": artefak,
                     "ada": [os.path.exists(f) for f in artefak],
                     "ukuran_KB": [round(os.path.getsize(f) / 1024, 1) if os.path.exists(f) else None for f in artefak]})
print(f"{int(ceks['ada'].sum())}/{len(ceks)} artefak tersedia")
ceks

#### Apa yang didapat
Tabel daftar berkas keluaran beserta status keberadaan dan ukurannya. Semua baris `ada = True` berarti notebook berjalan penuh dan siap dipakai dashboard.

### 7.2 Referensi
* Hansen, M. C. et al. (2013). *High-Resolution Global Maps of 21st-Century Forest Cover Change*. Science, 342.
* Harris, N. L. et al. (2021). *Global maps of twenty-first century forest carbon fluxes*. Nature Climate Change, 11.
* Global Forest Watch - dokumentasi data tree cover loss, drivers, dan forest carbon (gfw.org).
* Lundberg, S. M. & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
* Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR, 12.
* Lawson, C. L. & Hanson, R. J. (1995). *Solving Least Squares Problems* (algoritma NNLS). SIAM.
* Chapman, P. et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*.